# SkinSense — Train on all 3 datasets (12 classes)

Assembles **three sources** into one balanced ImageFolder, removes duplicate /
leaked images, then trains with the canonical recipe (`train.py:run_training`).

1. **Base set** — your merged `data.zip` (in Drive).
2. **SCIN** — Google's public dataset (auto-downloaded from GCS, diverse skin tones).
3. **Fitzpatrick17k** — `archive.zip` + `fitzpatrick17k.csv` (you upload to Drive).

Then `dedup_dataset.py` strips train/val leakage so the accuracy is **real**.

> A note on "100%": with de-dup in place you will NOT see 100%, and that's the
> point — a leak-free 12-class derm model realistically lands ~75-90% top-1 /
> ~95%+ top-3. If you ever saw ~100%, it was leaked duplicates, not learning.

**Colab (GPU):** Runtime → Change runtime type → GPU. Put `data.zip`,
`archive.zip`, and `fitzpatrick17k.csv` in `My Drive/SkinSense/`, then run top to bottom.

In [ ]:
# --- deps ---
!pip -q install torch torchvision gcsfs pandas pillow tqdm

In [ ]:
# --- Dataset 1: base set (your merged data.zip in Drive) ---
import os, zipfile, time
DRIVE_DIR = "/content/drive/MyDrive/SkinSense"   # <-- all 3 uploads live here

if not os.path.isdir("data"):
    if os.path.isdir("/content"):
        from google.colab import drive
        drive.mount("/content/drive")
        zip_path = f"{DRIVE_DIR}/data.zip"
        assert os.path.isfile(zip_path), f"not found: {zip_path}"
        print("unzipping base data.zip -> ./data …")
        t0=time.time()
        with zipfile.ZipFile(zip_path) as z: z.extractall(".")
        print(f"done in {time.time()-t0:.0f}s")
    else:
        raise FileNotFoundError("data/ not found (run the last cell to build data.zip).")
print("base files:", sum(len(f) for _,_,f in os.walk("data")))

In [ ]:
# --- get the canonical training + dataset scripts from the repo ---
import os, sys
REPO = "https://github.com/Mohammed-Ateeb/SkinSenseAi.git"   # public repo
# Private? use:  REPO = "https://<GITHUB_TOKEN>@github.com/Mohammed-Ateeb/SkinSenseAi.git"
if not os.path.isdir("SkinSenseAi"):
    !git clone -q $REPO SkinSenseAi
TRAIN_DIR = "SkinSenseAi/backend/app/ml/training"
sys.path.insert(0, "SkinSenseAi/backend/app/ml")   # model_loader
sys.path.insert(0, TRAIN_DIR)                        # train
from train import run_training
from model_loader import CLASS_NAMES
print("classes:", CLASS_NAMES)

In [ ]:
# --- Dataset 2: SCIN (Google, public GCS — no login) ---
# Downloads only the images it maps + caps per class, straight into ./data.
!python $TRAIN_DIR/prepare_scin.py --out ./data --per-class 1200 --min-weight 0.5

In [ ]:
# --- Dataset 3: Fitzpatrick17k (you uploaded archive.zip + csv to Drive) ---
import os, zipfile
FITZ_ZIP = f"{DRIVE_DIR}/archive.zip"
FITZ_CSV = f"{DRIVE_DIR}/fitzpatrick17k.csv"
assert os.path.isfile(FITZ_ZIP), f"upload archive.zip to {DRIVE_DIR}"
assert os.path.isfile(FITZ_CSV), f"upload fitzpatrick17k.csv to {DRIVE_DIR}"
if not os.path.isdir("fitz_images"):
    print("unzipping archive.zip -> ./fitz_images …")
    with zipfile.ZipFile(FITZ_ZIP) as z: z.extractall("fitz_images")
!python $TRAIN_DIR/prepare_fitzpatrick.py --csv "$FITZ_CSV" --images ./fitz_images --out ./data --per-class 1200

In [ ]:
# --- De-duplicate the merged set (removes train/val leakage) ---
# Dry-run first to SEE what would go, then apply.
!python $TRAIN_DIR/dedup_dataset.py --data ./data --near 4
!python $TRAIN_DIR/dedup_dataset.py --data ./data --near 4 --apply

# final per-class counts
import os
for split in ("train","val"):
    print(f"
{split}:")
    d=f"data/{split}"
    for c in sorted(os.listdir(d)):
        p=os.path.join(d,c)
        if os.path.isdir(p): print(f"  {c:22s} {len(os.listdir(p))}")

In [ ]:
# --- train on the merged, de-duplicated set ---
ARCH = "convnext_tiny"        # or "efficientnet_b3" / "efficientnet_b0"
OUT  = f"skinsense_{ARCH}.pt"
best_macro = run_training(
    data="./data", out=OUT, arch=ARCH,
    epochs=30, warmup_epochs=3, batch_size=32, lr=3e-4,
    mixup=0.2, balanced=True,
)
print("best val macro-recall:", best_macro)
import shutil, os
if os.path.isdir(DRIVE_DIR):
    shutil.copy(OUT, f"{DRIVE_DIR}/{OUT}"); print("copied", OUT, "-> Drive")

## Serve the checkpoint

Copy `skinsense_<arch>.pt` into `backend/weights/` and set:
```
WEIGHTS_PATH=./weights/skinsense_convnext_tiny.pt
```
Self-describing checkpoint — `load_model()` reads the arch + calibrated
temperature from the file, so no `MODEL_ARCH` needed. Keep the confidence
threshold + disclaimer: decision support, not a medical device.

In [ ]:
# --- (local helper) build data.zip from your BASE set, then upload to Drive ---
# Run LOCALLY where your base data/ lives. SCIN + Fitzpatrick are pulled in Colab,
# so this zip should contain only your base merged set.
import shutil
shutil.make_archive("data", "zip", ".", "data")
print("wrote data.zip -> upload to My Drive/SkinSense/")